In [1]:
# Ô 1 — Thiết lập và bảng cấp file
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path("..").resolve()))

from src.clean import load_events, clean_events

RAW = Path("..") / "data" / "raw"
ev = clean_events(load_events(RAW / "evtx_data.csv"))

files = (ev.groupby("EVTX_FileName")
          .agg(tactic=("EVTX_Tactic","first"),
               n_events=("EventID","size"),
               computer=("Computer","first"))
          .reset_index()
        )
files["tactic"] = files["tactic"].astype(str)
print(ev.shape,files.shape)



(4633, 16) (249, 4)


Phần A — Phân bố: trung bình hay trung vị?

In [2]:
# Ô 2 — Mô tả phân bố số bản ghi mỗi file
s = files["n_events"]

print(s.describe().round(1)) # in 8 chỉ số thống kê một lần. .round(1) làm tròn cho dễ đọc.
print("----------")
print(round(s.mean(),1), s.median(),round(s.skew(),2)) 
# độ lệch (skewness). Bằng 0 là đối xứng, dương là đuôi kéo dài bên phải. Giá trị 9 là lệch rất mạnh; thường trên 1 đã được coi là lệch nhiều.
print("----------")
# các phân vị. Ví dụ 0.9: 25.4 nghĩa là 90% số file có dưới khoảng 25 bản ghi. Khoảng cách từ 95% (53,8) lên 99% (296) cho thấy đuôi rất dài.
print(s.quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

count    249.0
mean      18.6
std       71.4
min        1.0
25%        3.0
50%        5.0
75%       10.0
max      869.0
Name: n_events, dtype: float64
----------
18.6 5.0 9.01
----------
0.25      3.00
0.50      5.00
0.75     10.00
0.90     25.40
0.95     53.80
0.99    296.28
Name: n_events, dtype: float64


Phần B — Mất cân bằng nhãn

In [ ]:
# Ô 3 — Mức độ mất cân bằng
counts = files["tactic"].value_counts()
pct = (counts / counts.sum() * 100).round(1)
print(pd.DataFrame({"n_files": counts, "pct": pct}).reset_index())

print(round(counts.max() / counts.min(),1)) 
# counts.max() / counts.min(): tỷ lệ mất cân bằng (imbalance ratio), lấy lớp đông nhất chia lớp hiếm nhất. Lớp đông gấp 12 lần lớp hiếm.

                 tactic  n_files   pct
0  Privilege Escalation       61  24.5
1      Lateral Movement       47  18.9
2       Defense Evasion       36  14.5
3     Credential Access       35  14.1
4             Execution       33  13.3
5           Persistence       22   8.8
6             Discovery       10   4.0
7   Command and Control        5   2.0
12.2


In [10]:
# Ô 4 — Mô hình “lười”: luôn đoán lớp đông nhất
majority = counts.idxmax() # counts.idxmax(): nhãn có số đếm lớn nhất,
files["pred_majority"] = majority
acc = (files["pred_majority"] == files["tactic"]).mean()
# (files["pred_majority"] == files["tactic"]): so sánh từng dòng, ra True khi đoán đúng
# .mean(): tỷ lệ True, tức accuracy.
print(majority, round(acc * 100,1), round(100 / len(counts),1))

Privilege Escalation 24.5 12.5


In [11]:
# Ô 5 — Precision, recall, F1 từng lớp (tự tính để hiểu bản chất)
def per_class(df, y_true, y_pred): # hàm nhận bảng, tên cột nhãn thật và tên cột nhãn dự đoán.
    rows = []
    # lặp qua từng tactic theo thứ tự chữ cái. Mỗi vòng xét lớp c như bài toán “là c” hay “không phải c”.
    for c in sorted(df[y_true].unique()):
        tp = ((df[y_true] == c) & (df[y_pred] == c)).sum()
        fp = ((df[y_true] != c) & (df[y_pred] == c)).sum()
        fn = ((df[y_true] == c) & (df[y_pred] != c)).sum()
        p = tp / (tp + fp) if tp + fp > 0 else 0.0
        r = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1 = 2 * p * r / (p + r) if p + r > 0 else 0.0
        rows.append({"tactic": c, "support": tp + fn, "tp": tp, "fp": fp, "fn": fn,
                     "precision": round(p, 3), "recall": round(r, 3), "f1": round(f1, 3)})
    return pd.DataFrame(rows).set_index("tactic")

rep = per_class(files, "tactic", "pred_majority")
print(rep)
#                       support  tp   fp  fn  precision  recall     f1
# Command and Control         5   0    0   5      0.000     0.0  0.000
# ...
# Privilege Escalation       61  61  188   0      0.245     1.0  0.394

macro = rep["f1"].mean()
weighted = (rep["f1"] * rep["support"]).sum() / rep["support"].sum()
print(round(macro, 3), round(weighted, 3))   # 0.049  0.097

                      support  tp   fp  fn  precision  recall     f1
tactic                                                              
Command and Control         5   0    0   5      0.000     0.0  0.000
Credential Access          35   0    0  35      0.000     0.0  0.000
Defense Evasion            36   0    0  36      0.000     0.0  0.000
Discovery                  10   0    0  10      0.000     0.0  0.000
Execution                  33   0    0  33      0.000     0.0  0.000
Lateral Movement           47   0    0  47      0.000     0.0  0.000
Persistence                22   0    0  22      0.000     0.0  0.000
Privilege Escalation       61  61  188   0      0.245     1.0  0.394
0.049 0.097


Phần C — Rò rỉ dữ liệu khi chia train/test

In [ ]:
# Ô 6 — Chia ngẫu nhiên theo bản ghi (cách SAI)
test_rec = ev.sample(frac=0.2, random_state=42)
# ev.sample(frac=0.2, random_state=42): lấy ngẫu nhiên 20% số dòng làm tập test
# random_state=42 cố định “hạt giống” ngẫu nhiên, nên mỗi lần chạy ra cùng một kết quả và người khác chạy lại được.
train_rec = ev.drop(test_rec.index) # bỏ các dòng đã vào test, phần còn lại làm train

shared = set(train_rec["EVTX_FileName"]) & set(test_rec["EVTX_FileName"]) # set(a) & set(b): phép giao tập hợp, ra các file xuất hiện ở cả hai tập.
n_test_files = test_rec["EVTX_FileName"].nunique()
print(len(train_rec), len(test_rec))                                  # 3706 927
print(n_test_files, len(shared), round(len(shared) / n_test_files * 100, 1))
# 160 151 94.4

3706 927
160 151 94.4


In [ ]:
# Ô 7 — Chia theo file và giữ tỷ lệ lớp (cách ĐÚNG)
test_files = (files.groupby("tactic")
                   .sample(frac=0.2, random_state=42)["EVTX_FileName"])
is_test = ev["EVTX_FileName"].isin(test_files)
train_g, test_g = ev[~is_test], ev[is_test]

print(len(test_files), len(train_g), len(test_g))   # 49 4315 318
print(len(set(train_g["EVTX_FileName"]) & set(test_g["EVTX_FileName"])))   # 0

in_test = files["EVTX_FileName"].isin(test_files)
pd.DataFrame({
    "all":   files["tactic"].value_counts(normalize=True),
    "train": files.loc[~in_test, "tactic"].value_counts(normalize=True),
    "test":  files.loc[in_test, "tactic"].value_counts(normalize=True),
}).mul(100).round(1)
#                        all  train  test
# Command and Control    2.0    2.0   2.0
# Privilege Escalation  24.5   24.5  24.5
# ...

files.loc[in_test, "tactic"].value_counts()["Command and Control"]   # 1

Phần D — Ngoại lệ: đánh dấu, không xóa

In [13]:
# Ô 8 — Tìm file lớn bằng IQR và z-score
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print(q1, q3, iqr, upper)   # 3.0 10.0 7.0 20.5

files["is_large_file"] = files["n_events"] > upper
n_large = files["is_large_file"].sum()
share = files.loc[files["is_large_file"], "n_events"].sum() / files["n_events"].sum() * 100
print(n_large, round(share, 1))   # 29  72.5

z = (s - s.mean()) / s.std()
print((z.abs() > 3).sum())        # 4
files.loc[z.abs() > 3, ["EVTX_FileName", "tactic", "n_events"]]
# LM_5145_Remote_FileCopy.evtx                  Lateral Movement      869
# Sysmon_UACME_64.evtx                          Privilege Escalation  530
# persistence_sysmon_11_13_1_shime_appfix.evtx  Persistence           237
# rogue_msi_url_1040_1042.evtx                  Execution             351

3.0 10.0 7.0 20.5
29 72.5
4


,EVTX_FileName,tactic,n_events
42,LM_5145_Remote_FileCopy.evtx,Lateral Movement,869
110,Sysmon_UACME_64.evtx,Privilege Escalation,530
176,persistence_sysmon_11_13_1_shime_appfix.evtx,Persistence,237
193,rogue_msi_url_1040_1042.evtx,Execution,351


In [14]:
# Ô 9 — Ngoại lệ ảnh hưởng kết quả thế nào
ev2 = ev.merge(files[["EVTX_FileName", "is_large_file"]], on="EVTX_FileName", how="left")

pd.DataFrame({
    "all":      ev2["EVTX_Tactic"].astype(str).value_counts(normalize=True),
    "no_large": ev2.loc[~ev2["is_large_file"], "EVTX_Tactic"].astype(str).value_counts(normalize=True),
}).mul(100).round(1)
#                        all  no_large
# Command and Control    9.3       2.4
# Lateral Movement      27.8      14.6
# Privilege Escalation  23.5      31.8
# ...

,all,no_large
EVTX_Tactic,,
Command and Control,9.3,2.4
Credential Access,6.1,12.2
Defense Evasion,9.3,13.4
Discovery,3.6,4.1
Execution,11.5,12.0
Lateral Movement,27.8,14.6
Persistence,8.9,9.5
Privilege Escalation,23.5,31.8


In [15]:
# Ô 10 — Cái hiếm thường đáng chú ý
pf = (ev.dropna(subset=["process_name"])
        .groupby("process_name")["EVTX_FileName"].nunique())
print(len(pf), (pf == 1).sum())   # 196 127

rare = pf[pf == 1].index
rare_df = (ev[ev["process_name"].isin(rare)]
             .groupby("process_name")
             .agg(file=("EVTX_FileName", "first"),
                  tactic=("EVTX_Tactic", "first"),
                  n=("EventID", "size")))
rare_df.loc[["sdbinst.exe", "poc.exe", "winsat.exe", "ipconfig.exe"]]
#               file                                           tactic                n
# sdbinst.exe   persistence_sysmon_11_13_1_shime_appfix.evtx   Persistence          47
# poc.exe       eop_appcontainer_il_broker_filewrite.evtx      Privilege Escalation  2
# winsat.exe    sysmon_11_7_1_uacbypass_windirectory_mocking   Privilege Escalation  3
# ipconfig.exe  LM_winrm_exec_sysmon_1_winrshost.evtx          Lateral Movement      1

196 127


,file,tactic,n
process_name,,,
sdbinst.exe,persistence_sysmon_11_13_1_shime_appfix.evtx,Persistence,47
poc.exe,eop_appcontainer_il_broker_filewrite.evtx,Privilege Escalation,2
winsat.exe,sysmon_11_7_1_uacbypass_windirectory_mocking.evtx,Privilege Escalation,3
ipconfig.exe,LM_winrm_exec_sysmon_1_winrshost.evtx,Lateral Movement,1


Phần E — Sai lệch do cách thu thập dữ liệu

In [16]:
# Ô 11 — Dữ liệu đến từ đâu
print(ev["Computer"].nunique())   # 19
print(ev["Computer"].value_counts().head(3))
# PC01.example.corp 1591, MSEDGEWIN10 888, IEWIN7 746
print(round(ev["Computer"].value_counts().head(3).sum() / len(ev) * 100, 1))   # 69.6

users = pd.read_csv(RAW / "evtx_data.csv", usecols=["SubjectUserName", "User"], low_memory=False)
print(users["User"].value_counts().head(3))
# NT AUTHORITY\SYSTEM 329, IEWIN7\IEUser 203, MSEDGEWIN10\IEUser 157

ev.groupby("EVTX_FileName")["ts"].min().dt.year.value_counts().sort_index()
# 2017: 2, 2018: 1, 2019: 158, 2020: 84, 2021: 4   (số file)

19
Computer
PC01.example.corp    1591
MSEDGEWIN10           888
IEWIN7                746
Name: count, dtype: int64
69.6
User
NT AUTHORITY\SYSTEM    329
IEWIN7\IEUser          203
MSEDGEWIN10\IEUser     157
Name: count, dtype: int64


ts
2017      2
2018      1
2019    158
2020     84
2021      4
Name: count, dtype: int64

Phần F — Phát hiện hay giả thuyết?

In [ ]:
# Ô 12 — Kiểm chứng: “EventID 5145 là dấu hiệu của Lateral Movement”?
e5145 = ev[ev["EventID"] == 5145]

print(len(e5145))   # 950
print(e5145["EVTX_Tactic"].astype(str).value_counts().to_dict())
# {'Lateral Movement': 929, 'Discovery': 15, 'Credential Access': 3, ...}

print(e5145["EVTX_FileName"].nunique())   # 14
e5145.groupby("EVTX_Tactic", observed=True)["EVTX_FileName"].nunique()
# Lateral Movement 7, Discovery 3, Credential Access 2, Defense Evasion 1, Privilege Escalation 1